# Data Loading & Exploration

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

PROC = ROOT / 'data' / 'processed'
SPLITS = ROOT / 'data' / 'splits'

plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

## 1. Language Identification

In [ ]:
lang_df = pd.read_csv(PROC / 'language_identification_cleaned.csv')
print(f'Shape: {lang_df.shape}')
print(f'Columns: {list(lang_df.columns)}')
lang_df.head(3)

In [ ]:
counts = lang_df['language'].value_counts()
fig, ax = plt.subplots(figsize=(12, 4))
counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Language Identification — Class Distribution', fontsize=13)
ax.set_xlabel('Language')
ax.set_ylabel('Sample count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
print(counts.to_frame('count').assign(pct=lambda d: (d['count']/len(lang_df)*100).round(1)))

In [ ]:
# Text length distribution
lang_df['word_count'] = lang_df['text'].str.split().apply(len)
lang_df['word_count'].describe().to_frame().T

## 2. Emotion Dataset

In [ ]:
emo_df = pd.read_csv(PROC / 'emotion_cleaned.csv')
print(f'Shape: {emo_df.shape}')
emo_df.head(3)

In [ ]:
emo_counts = emo_df['emotion'].value_counts()
colors = ['#e74c3c','#f39c12','#2ecc71','#3498db','#9b59b6','#1abc9c']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

emo_counts.plot(kind='bar', ax=axes[0], color=colors[:len(emo_counts)], edgecolor='white')
axes[0].set_title('Emotion — Class Distribution (bar)', fontsize=12)
axes[0].set_xlabel('')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

axes[1].pie(emo_counts, labels=emo_counts.index, autopct='%1.1f%%',
            colors=colors[:len(emo_counts)], startangle=90)
axes[1].set_title('Emotion — Class Distribution (pie)', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Mental Health Counseling

In [ ]:
mh_df = pd.read_csv(PROC / 'mental_health_cleaned.csv')
print(f'Shape: {mh_df.shape}')
mh_df.head(2)

In [ ]:
mh_df['ctx_words'] = mh_df['context'].str.split().apply(len)
mh_df['resp_words'] = mh_df['response'].str.split().apply(len)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, title in zip(axes,
    ['ctx_words', 'resp_words'],
    ['Context length (words)', 'Response length (words)']):
    ax.hist(mh_df[col], bins=40, color='teal', edgecolor='white')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Words')
    ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

mh_df[['ctx_words','resp_words']].describe().round(1)

## 4. Verify Splits

In [ ]:
for name, col in [('language', 'language'), ('emotion', 'emotion')]:
    dfs = {s: pd.read_csv(SPLITS / f'{name}_{s}.csv') for s in ['train','val','test']}
    sizes = {s: len(df) for s, df in dfs.items()}
    total = sum(sizes.values())
    print(f'\n{name.upper()} split sizes: {sizes}  (total={total:,})')
    
    # Check all classes present
    all_labels = set(dfs['train'][col].unique())
    for split, df in dfs.items():
        missing = all_labels - set(df[col].unique())
        status = 'ok' if not missing else f'✗ missing {missing}'
        print(f'  {split:<6}: {status}')

In [ ]:
# Visual: split distribution comparison for language
lang_splits = {s: pd.read_csv(SPLITS / f'language_{s}.csv') for s in ['train','val','test']}
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for ax, (split, df) in zip(axes, lang_splits.items()):
    df['language'].value_counts().sort_index().plot(kind='bar', ax=ax,
        color='steelblue', edgecolor='white')
    ax.set_title(f'Language — {split}  (n={len(df):,})', fontsize=11)
    ax.tick_params(axis='x', rotation=60, labelsize=7)
plt.suptitle('Stratified split: language distribution', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 5. Knowledge Base Summary

In [ ]:
kb_path = PROC / 'knowledge_base_combined.json'
if kb_path.exists():
    with open(kb_path) as f:
        kb = json.load(f)
    kb_df = pd.DataFrame(kb)
    print(f'Total chunks: {len(kb_df):,}')
    print('\nBy source type:')
    print(kb_df['source_type'].value_counts().to_string())
    print('\nToken stats:')
    print(kb_df['tokens'].describe().round(1).to_string())
    kb_df.head(3)
else:
    print('knowledge_base_combined.json not found — run 00_prepare_data.py first.')

In [ ]:
if 'kb_df' in dir():
    fig, ax = plt.subplots(figsize=(9, 3))
    kb_df['tokens'].clip(upper=600).hist(bins=40, ax=ax, color='coral', edgecolor='white')
    ax.set_title('Knowledge Base — Token distribution per chunk (capped at 600)', fontsize=12)
    ax.set_xlabel('Estimated tokens')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()